# Setup

## Libraries

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

ModuleNotFoundError: No module named 'dotenv'

In [ ]:
import pandas as pd

In [ ]:
from collections import Counter

In [ ]:
import numpy as np

In [ ]:
from gensim.models import Word2Vec

In [ ]:
import requests

## Environment

In [ ]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
REPO_ROOT = PROJECT_ROOT.parent

In [ ]:
load_dotenv(REPO_ROOT / ".env")
TMDB_API_KEY = os.getenv("TMDB_API_KEY")

In [ ]:
MOVIES_SMALL_DIR = PROJECT_ROOT / "data/ml-latest-small"
MOVIES_32M_DIR = PROJECT_ROOT / "data/ml-32m"

## Data

### Read Data

In [ ]:
links_df = pd.read_csv(MOVIES_SMALL_DIR / "links.csv")
movies_df = pd.read_csv(MOVIES_SMALL_DIR / "links.csv")
ratings_df = pd.read_csv(MOVIES_SMALL_DIR / "ratings.csv")
tags_df = pd.read_csv(MOVIES_SMALL_DIR / "links.csv")

### Map into Sequence

In [ ]:
item_sequence = (
    ratings_df
    .sort_values(['userId','timestamp'])
    .groupby('userId')['movieId']
    .agg(list)
)

In [ ]:
sentences = [[str(i) for i in seq] for seq in item_sequence]

# EDA

In [ ]:
# Check if a user interacts with/ rates an item more than once
(ratings_df.groupby(['userId', 'movieId'])['timestamp'].agg('count') > 1).sum()
## -> No duplicate interaction pairs

np.int64(0)

In [ ]:
ratings_df

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


## Fetch Info from API

In [ ]:
links_df

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0
...,...,...,...
9737,193581,5476944,432131.0
9738,193583,5914996,445030.0
9739,193585,6397426,479308.0
9740,193587,8391976,483455.0


In [ ]:
links = pd.read_csv("links.csv")

def get_poster(tmdb_id):
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}"
    params = {"api_key": TMDB_API_KEY}

    r = requests.get(url, params=params).json()

    poster = r.get("poster_path")
    if poster:
        return f"https://image.tmdb.org/t/p/w500{poster}"
    return None

{"status_code":7,"status_message":"Invalid API key: You must be granted a valid key.","success":false}



In [ ]:
test_id = 862
get_poster(test_id)

In [ ]:
links["poster_url"] = links["tmdbId"].apply(get_poster)

In [ ]:
# Parse posters from URLs and save

links = links.dropna(subset=["poster_url", "tmdbId"])
links = links.drop_duplicates("tmdbId")

POSTERS_DIR = PROJECT_ROOT / "data" / "ml-small-posters"
POSTERS_DIR.mkdir(exist_ok=True)

movie_tmdb_ids = links['tmdbId'].unique().tolist()

for _, row in links.iterrows():

    tmdb_id = row["tmdbId"]
    poster_url = row["poster_url"]

    # skip invalid entries
    if pd.isna(tmdb_id) or poster_url is None:
        continue

    try:
        response = requests.get(poster_url, timeout=10)

        if response.status_code != 200:
            continue

        poster_path = POSTERS_DIR / f"{int(tmdb_id)}.jpg"

        with open(poster_path, "wb") as f:
            f.write(response.content)

    except Exception as e:
        print(f"Failed {tmdb_id}: {e}")